# Standard conformal response units

A response transformation changes the coordinates in which a model is fitted. Conformal calibration needs its predictions and observed outcomes in common units. This diagnostic compares standard `glmboost` and `gamboost` split, CV+ and CQR intervals with independent references constructed in the original response units.

The [protocol](conformal_response_units/PROTOCOL.md) fixes 112 cases with prescribed stopping, eight tuned cases, and 12 deliberately unsupported tuning calls. All cases use one deterministic fixture: 240 observations, seed 636, $x$ uniform on $[-1,1]$ and $Y=\exp(0.3+0.6x+0.2Z)$ with standard normal $Z$. The response has arbitrary simulation units. Methods share this fixture; there are no independent sampling replicates, coverage estimates or Monte Carlo standard errors.

In [1]:
from pathlib import Path
import json
import sys

import numpy as np
import polars as pl
from great_tables import GT, md

root = next(p for p in (Path.cwd(), *Path.cwd().parents) if (p / "Cargo.toml").exists())
sys.path.insert(0, str(root / "python"))
sys.path.insert(0, str(root))
import mboostrs as mb
from benches.audit_conformal_response_units import inputs
from benches.summarize_conformal_response_units import load, validate
assert Path(mb.__file__).resolve().is_relative_to(root)

before, after = load("baseline"), load("corrected")
rows, affine, facts = validate(before, after)

In [2]:
comparison = pl.DataFrame([{
    "Method": r["method"], "Calibration": r["route"], "Cases": r["cases"],
    "Before failures": r["baseline_failures"], "Before disagreements": r["baseline_disagreements"],
    "Before delta": f'{r["baseline_max_error"]:.3e}',
    "Corrected delta": f'**{r["corrected_max_error"]:.3e}**'} for r in rows])
(GT(comparison)
 .tab_header(title="Agreement with independent original-response references",
             subtitle="Maximum absolute point, endpoint or correction discrepancy in Y units; smaller is better")
 .tab_spanner(label="Distance from the original-unit contract", columns=["Before delta", "Corrected delta"])
 .fmt_markdown(columns="Corrected delta")
 .tab_source_note(md("Bold values satisfy the prescribed 1e-10 tolerance. Cases cross two frontends, seven "
    "response specifications and query/training-row targets. Before disagreements count completed cases "
    "above tolerance; failures count exceptions separately. Baseline deltas include a change in reported units "
    "and are not all calibration defects. Twelve unsupported tuning calls are reported below. "
    "Source: baseline.json.gz and corrected.json.gz; runner: benches/audit_conformal_response_units.py. "
    "No sampling uncertainty was estimated.")))

GT(_tbl_data=shape: (4, 7)
┌────────┬─────────────┬───────┬──────────┬───────────────┬──────────────┬─────────────────┐
│ Method ┆ Calibration ┆ Cases ┆ Before   ┆ Before        ┆ Before delta ┆ Corrected delta │
│ ---    ┆ ---         ┆ ---   ┆ failures ┆ disagreements ┆ ---          ┆ ---             │
│ str    ┆ str         ┆ i64   ┆ ---      ┆ ---           ┆ str          ┆ str             │
│        ┆             ┆       ┆ i64      ┆ i64           ┆              ┆                 │
╞════════╪═════════════╪═══════╪══════════╪═══════════════╪══════════════╪═════════════════╡
│ split  ┆ internal    ┆ 28    ┆ 0        ┆ 24            ┆ 1.038e+01    ┆ **0.000e+00**   │
│ split  ┆ external    ┆ 28    ┆ 24       ┆ 0             ┆ 0.000e+00    ┆ **1.332e-15**   │
│ cv+    ┆ internal    ┆ 32    ┆ 0        ┆ 26            ┆ 1.076e+01    ┆ **0.000e+00**   │
│ cqr    ┆ internal    ┆ 32    ┆ 0        ┆ 26            ┆ 9.925e+00    ┆ **6.661e-16**   │
└────────┴─────────────┴───────┴──────────┴───────────────┴──────────────┴─────────────────┘, _body=<great_tables._gt_data.Body object at 0x10f871790>, _boxhead=Boxhead([ColInfo(var='Method', type=<ColInfoTypeEnum.default: 1>, column_label='Method', column_align='left', column_width=None), ColInfo(var='Calibration', type=<ColInfoTypeEnum.default: 1>, column_label='Calibration', column_align='left', column_width=None), ColInfo(var='Cases', type=<ColInfoTypeEnum.default: 1>, column_label='Cases', column_align='right', column_width=None), ColInfo(var='Before failures', type=<ColInfoTypeEnum.default: 1>, column_label='Before failures', column_align='right', column_width=None), ColInfo(var='Before disagreements', type=<ColInfoTypeEnum.default: 1>, column_label='Before disagreements', column_align='right', column_width=None), ColInfo(var='Before delta', type=<ColInfoTypeEnum.default: 1>, column_label='Before delta', column_align='left', column_width=None), ColInfo(var='Corrected delta', type=<ColInfoTypeEnum.default: 1>, column_label='Corrected delta', column_align='left', column_width=None)]), _stub=<great_tables._gt_data.Stub object at 0x10f3214f0>, _spanners=Spanners([SpannerInfo(spanner_id='Distance from the original-unit contract', spanner_level=0, spanner_label='Distance from the original-unit contract', spanner_units=None, spanner_pattern=None, vars=['Before delta', 'Corrected delta'], built=None)]), _heading=Heading(title='Agreement with independent original-response references', subtitle='Maximum absolute point, endpoint or correction discrepancy in Y units; smaller is better', preheader=None), _stubhead=None, _summary_rows=<great_tables._gt_data.SummaryRows object at 0x10c466c00>, _summary_rows_grand=<great_tables._gt_data.SummaryRows object at 0x10f321370>, _source_notes=[Md(text='Bold values satisfy the prescribed 1e-10 tolerance. Cases cross two frontends, seven response specifications and query/training-row targets. Before disagreements count completed cases above tolerance; failures count exceptions separately. Baseline deltas include a change in reported units and are not all calibration defects. Twelve unsupported tuning calls are reported below. Source: baseline.json.gz and corrected.json.gz; runner: benches/audit_conformal_response_units.py. No sampling uncertainty was estimated.')], _footnotes=[], _styles=[], _locale=<great_tables._gt_data.Locale object at 0x10ee012b0>, _formats=[<great_tables._gt_data.FormatInfo object at 0x10f84daf0>], _substitutions=[], _col_merge=[], _transforms=[], _options=Options(table_id=OptionsInfo(scss=False, category='table', type='value', value=None), table_caption=OptionsInfo(scss=False, category='table', type='value', value=None), table_width=OptionsInfo(scss=True, category='table', type='px', value='auto'), table_layout=OptionsInfo(scss=True, category='table', type='value', value='fixed'), table_margin_left=OptionsInfo(scss=True, category='table', type='px', value='auto'), table_margin_right=OptionsInfo(scss=True, category='table', type='px', 

All {eval}`facts["supported_cases"]` supported cases agree with the reference within {eval}`facts["maximum_corrected_error"]` original-response units. The baseline has {eval}`facts["baseline_failures"]` exceptions and {eval}`facts["baseline_disagreements"]` completed cases whose outputs differ from the original-unit reference. The latter count includes output-unit differences: it does not imply that every former transformed-unit interval had incorrect calibration.

## Response coordinates

For each proper-training fit $b$, let $g_b$ denote its fitted response transform and $\hat f_b(x)$ its response prediction in transformed units. The point function used for calibration is $p_b(x)=g_b^{-1}(\hat f_b(x))$. Absolute scores are $|Y_i-p_b(X_i)|$. Each CV+ residual and query prediction uses its own fold's inverse. CQR inverse-transforms its two candidate functions, orders them, then computes $\max(\ell(X_i)-Y_i,Y_i-u(X_i))$. Its correction and endpoints therefore have the same units as $Y$.

The inverse of a fitted transformed location generally differs from the original-response mean. `predicted` is the point function for split/CV+ and the calibrated-band midpoint for CQR; `fitted_mean` is unavailable for transformed fits. For nonlinear transforms, calibrating an original-unit score also differs from inverse-transforming the endpoints of a transformed-unit interval. A candidate outside an inverse transform's domain raises a descriptive error.

The following example fixes `mstop=30` before calibration. `scale(y)` learns a response mean and sample standard deviation, and each internal refit relearns them on its own proper-training rows. Standard `predict(type="response")` continues to report the fitted transformed response; the conformal result reports original $Y$ units.

In [3]:
frame, query = inputs()
model = mb.gamboost("scale(y) ~ bols(x)", frame, control=mb.boost_control(60))
interval = mb.conformal_interval(model, query, method="cqr", seed=4, mstop=30, level=.9)
assert interval.fitted_mean is None

preview = pl.DataFrame({"x": query["x"], "Lower": interval.lower,
                        "Point": interval.predicted, "Upper": interval.upper})
(GT(preview)
 .tab_header(title="CQR with a fitted response scale", subtitle="Original Y units; nominal marginal level 90%")
 .fmt_number(columns=["Lower", "Point", "Upper"], decimals=6)
 .tab_source_note("The point is the midpoint of the calibrated band, not an estimated original-response mean. "
                  "The specification and stopping count must be chosen without calibration responses."))

GT(_tbl_data=shape: (3, 4)
┌──────┬──────────┬──────────┬──────────┐
│ x    ┆ Lower    ┆ Point    ┆ Upper    │
│ ---  ┆ ---      ┆ ---      ┆ ---      │
│ f64  ┆ f64      ┆ f64      ┆ f64      │
╞══════╪══════════╪══════════╪══════════╡
│ -0.6 ┆ 0.387387 ┆ 0.969415 ┆ 1.551444 │
│ 0.0  ┆ 0.664941 ┆ 1.327113 ┆ 1.989286 │
│ 0.6  ┆ 0.942496 ┆ 1.684811 ┆ 2.427127 │
└──────┴──────────┴──────────┴──────────┘, _body=<great_tables._gt_data.Body object at 0x10f067d70>, _boxhead=Boxhead([ColInfo(var='x', type=<ColInfoTypeEnum.default: 1>, column_label='x', column_align='right', column_width=None), ColInfo(var='Lower', type=<ColInfoTypeEnum.default: 1>, column_label='Lower', column_align='right', column_width=None), ColInfo(var='Point', type=<ColInfoTypeEnum.default: 1>, column_label='Point', column_align='right', column_width=None), ColInfo(var='Upper', type=<ColInfoTypeEnum.default: 1>, column_label='Upper', column_align='right', column_width=None)]), _stub=<great_tables._gt_data.Stub object at 0x10edbc1a0>, _spanners=Spanners([]), _heading=Heading(title='CQR with a fitted response scale', subtitle='Original Y units; nominal marginal level 90%', preheader=None), _stubhead=None, _summary_rows=<great_tables._gt_data.SummaryRows object at 0x10f84cc20>, _summary_rows_grand=<great_tables._gt_data.SummaryRows object at 0x110777530>, _source_notes=['The point is the midpoint of the calibrated band, not an estimated original-response mean. The specification and stopping count must be chosen without calibration responses.'], _footnotes=[], _styles=[], _locale=<great_tables._gt_data.Locale object at 0x1107770b0>, _formats=[<great_tables._gt_data.FormatInfo object at 0x10f2f1190>], _substitutions=[], _col_merge=[], _transforms=[], _options=Options(table_id=OptionsInfo(scss=False, category='table', type='value', value=None), table_caption=OptionsInfo(scss=False, category='table', type='value', value=None), table_width=OptionsInfo(scss=True, category='table', type='px', value='auto'), table_layout=OptionsInfo(scss=True, category='table', type='value', value='fixed'), table_margin_left=OptionsInfo(scss=True, category='table', type='px', value='auto'), table_margin_right=OptionsInfo(scss=True, category='table', type='px', value='auto'), table_background_color=OptionsInfo(scss=True, category='table', type='value', value='#FFFFFF'), table_additional_css=OptionsInfo(scss=False, category='table', type='values', value=[]), table_font_names=OptionsInfo(scss=False, category='table', type='values', value=['-apple-system', 'BlinkMacSystemFont', 'Segoe UI', 'Roboto', 'Oxygen', 'Ubuntu', 'Cantarell', 'Helvetica Neue', 'Fira Sans', 'Droid Sans', 'Arial', 'sans-serif']), table_font_size=OptionsInfo(scss=True, category='table', type='px', value='16px'), table_font_weight=OptionsInfo(scss=True, category='table', type='value', value='normal'), table_font_style=OptionsInfo(scss=True, category='table', type='value', value='normal'), table_font_color=OptionsInfo(scss=True, category='table', type='value', value='#333333'), table_font_color_light=OptionsInfo(scss=True, category='table', type='value', value='#FFFFFF'), table_border_top_include=OptionsInfo(scss=False, category='table', type='boolean', value=True), table_border_top_style=OptionsInfo(scss=True, category='table', type='value', value='solid'), table_border_top_width=OptionsInfo(scss=True, category='table', type='px', value='2px'), table_border_top_color=OptionsInfo(scss=True, category='table', type='value', value='#A8A8A8'), table_border_right_style=OptionsInfo(scss=True, category='table', type='value', value='none'), table_border_right_width=OptionsInfo(scss=True, category='table', type='px', value='2px'), table_border_right_color=OptionsInfo(scss=True, category='table', type='value', value='#D3D3D3'), table_border_bottom_include=OptionsInfo(scss=False, category='table', type='boolean', value=True), table_border_bottom_style=OptionsInfo(scss=True, category='table', type='value', value='solid'), table_border_bo

## Affine calibration discrepancy

Centering and scaling are affine transformations, so transforming a band's endpoints back to the original units preserves its score construction up to the scale factor. The following diagnostic converts the baseline split/CQR endpoints with the corresponding proper-training mean and standard deviation before comparing them. A remaining discrepancy isolates the mixing of full-data calibration targets and proper-training response coordinates; changing output units alone cannot remove it. CV+ has several fold-specific transforms and is assessed directly against its original-response reference above.

In [4]:
affine_rows = []
for transform in ("center", "scale"):
    for method in ("split", "cqr"):
        selected = [r for r in affine if r["transform"] == transform and r["method"] == method]
        affine_rows.append({"Transform": transform, "Method": method, "Cases": len(selected),
            "Converted baseline": max(r["baseline_after_affine_unit_conversion"] for r in selected),
            "Corrected": f'**{max(r["corrected_error"] for r in selected):.3e}**'})
(GT(pl.DataFrame(affine_rows))
 .tab_header(title="Calibration error after affine unit conversion",
             subtitle="Maximum endpoint error in original Y units; smaller is better")
 .fmt_number(columns="Converted baseline", decimals=6)
 .fmt_markdown(columns="Corrected")
 .tab_source_note("Each row combines two frontends and two target frames. Baseline endpoints are first mapped "
                  "with the proper-training response parameters; corrected endpoints already use original units. "
                  "Bold values meet 1e-10. Source: affine_diagnostic.json, derived from the full archives."))

GT(_tbl_data=shape: (4, 5)
┌───────────┬────────┬───────┬────────────────────┬───────────────┐
│ Transform ┆ Method ┆ Cases ┆ Converted baseline ┆ Corrected     │
│ ---       ┆ ---    ┆ ---   ┆ ---                ┆ ---           │
│ str       ┆ str    ┆ i64   ┆ f64                ┆ str           │
╞═══════════╪════════╪═══════╪════════════════════╪═══════════════╡
│ center    ┆ split  ┆ 4     ┆ 0.024959           ┆ **0.000e+00** │
│ center    ┆ cqr    ┆ 4     ┆ 0.0828             ┆ **0.000e+00** │
│ scale     ┆ split  ┆ 4     ┆ 0.076488           ┆ **0.000e+00** │
│ scale     ┆ cqr    ┆ 4     ┆ 0.152724           ┆ **0.000e+00** │
└───────────┴────────┴───────┴────────────────────┴───────────────┘, _body=<great_tables._gt_data.Body object at 0x10f8a63c0>, _boxhead=Boxhead([ColInfo(var='Transform', type=<ColInfoTypeEnum.default: 1>, column_label='Transform', column_align='left', column_width=None), ColInfo(var='Method', type=<ColInfoTypeEnum.default: 1>, column_label='Method', column_align='left', column_width=None), ColInfo(var='Cases', type=<ColInfoTypeEnum.default: 1>, column_label='Cases', column_align='right', column_width=None), ColInfo(var='Converted baseline', type=<ColInfoTypeEnum.default: 1>, column_label='Converted baseline', column_align='right', column_width=None), ColInfo(var='Corrected', type=<ColInfoTypeEnum.default: 1>, column_label='Corrected', column_align='left', column_width=None)]), _stub=<great_tables._gt_data.Stub object at 0x113e20050>, _spanners=Spanners([]), _heading=Heading(title='Calibration error after affine unit conversion', subtitle='Maximum endpoint error in original Y units; smaller is better', preheader=None), _stubhead=None, _summary_rows=<great_tables._gt_data.SummaryRows object at 0x10efb86e0>, _summary_rows_grand=<great_tables._gt_data.SummaryRows object at 0x10c1fff80>, _source_notes=['Each row combines two frontends and two target frames. Baseline endpoints are first mapped with the proper-training response parameters; corrected endpoints already use original units. Bold values meet 1e-10. Source: affine_diagnostic.json, derived from the full archives.'], _footnotes=[], _styles=[], _locale=<great_tables._gt_data.Locale object at 0x1107775c0>, _formats=[<great_tables._gt_data.FormatInfo object at 0x110777fb0>, <great_tables._gt_data.FormatInfo object at 0x10c224a40>], _substitutions=[], _col_merge=[], _transforms=[], _options=Options(table_id=OptionsInfo(scss=False, category='table', type='value', value=None), table_caption=OptionsInfo(scss=False, category='table', type='value', value=None), table_width=OptionsInfo(scss=True, category='table', type='px', value='auto'), table_layout=OptionsInfo(scss=True, category='table', type='value', value='fixed'), table_margin_left=OptionsInfo(scss=True, category='table', type='px', value='auto'), table_margin_right=OptionsInfo(scss=True, category='table', type='px', value='auto'), table_background_color=OptionsInfo(scss=True, category='table', type='value', value='#FFFFFF'), table_additional_css=OptionsInfo(scss=False, category='table', type='values', value=[]), table_font_names=OptionsInfo(scss=False, category='table', type='values', value=['-apple-system', 'BlinkMacSystemFont', 'Segoe UI', 'Roboto', 'Oxygen', 'Ubuntu', 'Cantarell', 'Helvetica Neue', 'Fira Sans', 'Droid Sans', 'Arial', 'sans-serif']), table_font_size=OptionsInfo(scss=True, category='table', type='px', value='16px'), table_font_weight=OptionsInfo(scss=True, category='table', type='value', value='normal'), table_font_style=OptionsInfo(scss=True, category='table', type='value', value='normal'), table_font_color=OptionsInfo(scss=True, category='table', type='value', value='#333333'), table_font_color_light=OptionsInfo(scss=True, category='table', type='value', value='#FFFFFF'), table_border_top_include=OptionsInfo(scss=False, category='table', type='boolean', value=True), table_border_top_style=OptionsInfo(scss=True, category='table', type='value', value='solid'), table_border_top_widt

(tuning-contract)=
## Response transforms and inner tuning

Supported tuned cases use bare or log responses and five inner folds with `preprocessing="refit"`. The existing search starts at 1600 iterations, doubles at a boundary optimum, and stops at a cap of 10000. Each curve retains its exact positive-grid argmin; the archive check compares risk arrays within 1e-12 without substituting a different stopping rule.

Inner CV with `center(y)` or `scale(y)` remains unsupported because fold-specific family risks can have different response scales. These calls explicitly refuse; CQR with either transform requires an integer `mstop`. Split/CV+ also accept `None` for a prescribed existing path. Comparison of fold-specific tuning losses remains tracked in [issue #652](https://github.com/fcela/mboostrs/issues/652).

In [5]:
searches = []
for record in after["results"]:
    if record["budget"] != "tune":
        continue
    searches.append({"Frontend": record["frontend"].upper(), "Method": record["method"],
        "Response": record["transform"], "Status": record["status"],
        "Selected stops": ", ".join(str(s["optimum"]) for s in record["inner_cv"]) or "unavailable"})
(GT(pl.DataFrame(searches))
 .tab_header(title="Tuned calls and explicit refusals", subtitle="Eight supported calls and twelve named refusals")
 .tab_source_note("Stops list actual inner-CV optima in recorded order, including repeated search budgets. "
                  "Refusals contain the scale-changes-between-folds explanation. Full curves, raw row identities "
                  "and direct-reference stops remain in corrected.json.gz."))

GT(_tbl_data=shape: (20, 5)
┌──────────┬────────┬──────────┬─────────┬──────────────────┐
│ Frontend ┆ Method ┆ Response ┆ Status  ┆ Selected stops   │
│ ---      ┆ ---    ┆ ---      ┆ ---     ┆ ---              │
│ str      ┆ str    ┆ str      ┆ str     ┆ str              │
╞══════════╪════════╪══════════╪═════════╪══════════════════╡
│ GLM      ┆ cqr    ┆ bare     ┆ ok      ┆ 111, 148         │
│ GLM      ┆ cqr    ┆ log      ┆ ok      ┆ 54, 58           │
│ GLM      ┆ cv+    ┆ bare     ┆ ok      ┆ 182, 59, 182, 61 │
│ GLM      ┆ cv+    ┆ log      ┆ ok      ┆ 182, 60, 182, 65 │
│ GAM      ┆ cqr    ┆ bare     ┆ ok      ┆ 180, 124         │
│ …        ┆ …      ┆ …        ┆ …       ┆ …                │
│ GAM      ┆ split  ┆ scale    ┆ refused ┆ unavailable      │
│ GAM      ┆ cv+    ┆ center   ┆ refused ┆ unavailable      │
│ GAM      ┆ cv+    ┆ scale    ┆ refused ┆ unavailable      │
│ GAM      ┆ cqr    ┆ center   ┆ refused ┆ unavailable      │
│ GAM      ┆ cqr    ┆ scale    ┆ refused ┆ unavailable      │
└──────────┴────────┴──────────┴─────────┴──────────────────┘, _body=<great_tables._gt_data.Body object at 0x110777b30>, _boxhead=Boxhead([ColInfo(var='Frontend', type=<ColInfoTypeEnum.default: 1>, column_label='Frontend', column_align='left', column_width=None), ColInfo(var='Method', type=<ColInfoTypeEnum.default: 1>, column_label='Method', column_align='left', column_width=None), ColInfo(var='Response', type=<ColInfoTypeEnum.default: 1>, column_label='Response', column_align='left', column_width=None), ColInfo(var='Status', type=<ColInfoTypeEnum.default: 1>, column_label='Status', column_align='left', column_width=None), ColInfo(var='Selected stops', type=<ColInfoTypeEnum.default: 1>, column_label='Selected stops', column_align='left', column_width=None)]), _stub=<great_tables._gt_data.Stub object at 0x113e21220>, _spanners=Spanners([]), _heading=Heading(title='Tuned calls and explicit refusals', subtitle='Eight supported calls and twelve named refusals', preheader=None), _stubhead=None, _summary_rows=<great_tables._gt_data.SummaryRows object at 0x10c224140>, _summary_rows_grand=<great_tables._gt_data.SummaryRows object at 0x10f04eb10>, _source_notes=['Stops list actual inner-CV optima in recorded order, including repeated search budgets. Refusals contain the scale-changes-between-folds explanation. Full curves, raw row identities and direct-reference stops remain in corrected.json.gz.'], _footnotes=[], _styles=[], _locale=<great_tables._gt_data.Locale object at 0x10efba600>, _formats=[], _substitutions=[], _col_merge=[], _transforms=[], _options=Options(table_id=OptionsInfo(scss=False, category='table', type='value', value=None), table_caption=OptionsInfo(scss=False, category='table', type='value', value=None), table_width=OptionsInfo(scss=True, category='table', type='px', value='auto'), table_layout=OptionsInfo(scss=True, category='table', type='value', value='fixed'), table_margin_left=OptionsInfo(scss=True, category='table', type='px', value='auto'), table_margin_right=OptionsInfo(scss=True, category='table', type='px', value='auto'), table_background_color=OptionsInfo(scss=True, category='table', type='value', value='#FFFFFF'), table_additional_css=OptionsInfo(scss=False, category='table', type='values', value=[]), table_font_names=OptionsInfo(scss=False, category='table', type='values', value=['-apple-system', 'BlinkMacSystemFont', 'Segoe UI', 'Roboto', 'Oxygen', 'Ubuntu', 'Cantarell', 'Helvetica Neue', 'Fira Sans', 'Droid Sans', 'Arial', 'sans-serif']), table_font_size=OptionsInfo(scss=True, category='table', type='px', value='16px'), table_font_weight=OptionsInfo(scss=True, category='table', type='value', value='normal'), table_font_style=OptionsInfo(scss=True, category='table', type='value', value='normal'), table_font_color=OptionsInfo(scss=True, category='table', type='value', value='#333333'), table_font_color_light=OptionsInfo(scss=True, category='table', type='value', value='#FFFFFF'), table_border_top_include

## Evidence and reproduction

The matrix crosses two frontends, seven response specifications (bare, center, scale, log, log1p, sqrt and exp), three internal methods and query/training targets. External split calibration adds two frontends by seven specifications by two targets. Fixed cases request iteration 30 of a 60-step path. Split/CQR use seed 4 and 120 calibration rows; CV+ uses the same seed and four equal folds. Query predictors are -0.6, 0 and 0.6. External `newdata=None` refers to the proper model's 120 training rows.

The independent reference fits the same public models directly on raw proper-training rows. It uses explicit inverse formulas from those rows and NumPy order statistics for conformal arithmetic. It never calls private conformal prediction or response-inverse helpers. Production continues to use the Rust conformal core and the existing response-transform API. The archive comparison verifies identical input hashes, reference arrays, partitions and native identity across phases. Additional regression tests cover response/predictor mutation, imputation, omitted rows, inverse-domain errors, unequal folds, one-row chunks and paths longer than the original fit.

The complete compressed archives retain every exception and every array. Their generic evidence-helper uncertainty field contains Monte Carlo boilerplate ([issue #655](https://github.com/fcela/mboostrs/issues/655)); the frozen protocol and `validation.json` explicitly limit interpretation to this one deterministic fixture. There are no coverage, speed or competitor-ranking conclusions from these numerical checks.

In [6]:
example = next(r for r in after["results"] if r["frontend"] == "gam"
               and r["transform"] == "scale" and r["method"] == "cqr" and r["target"] == "query")
record = {"Frontend / response / method": "GAM / scale / CQR",
          "Proper-training rows": len(example["reference"]["partitions"]["training_rows"]),
          "Calibration rows": len(example["reference"]["partitions"]["calibration_rows"]),
          "Chosen stops": str(example["reference"]["stops"]),
          "Correction in Y units": example["quantile"],
          "Maximum discrepancy": example["max_absolute_error"]}
(GT(pl.DataFrame([{"Field": k, "Value": str(v)} for k, v in record.items()]))
 .tab_header(title="Representative complete record", subtitle="Fields used to verify one supported result")
 .tab_source_note("The archive also stores lower, upper and point arrays, raw row identities, data hashes, "
                  "warnings and native/source identities. This table selects a record before aggregation; "
                  "validation recomputes every discrepancy from the retained arrays."))

GT(_tbl_data=shape: (6, 2)
┌──────────────────────────────┬─────────────────────┐
│ Field                        ┆ Value               │
│ ---                          ┆ ---                 │
│ str                          ┆ str                 │
╞══════════════════════════════╪═════════════════════╡
│ Frontend / response / method ┆ GAM / scale / CQR   │
│ Proper-training rows         ┆ 120                 │
│ Calibration rows             ┆ 120                 │
│ Chosen stops                 ┆ [30, 30]            │
│ Correction in Y units        ┆ 0.30825297023630327 │
│ Maximum discrepancy          ┆ 0.0                 │
└──────────────────────────────┴─────────────────────┘, _body=<great_tables._gt_data.Body object at 0x10f8a4590>, _boxhead=Boxhead([ColInfo(var='Field', type=<ColInfoTypeEnum.default: 1>, column_label='Field', column_align='left', column_width=None), ColInfo(var='Value', type=<ColInfoTypeEnum.default: 1>, column_label='Value', column_align='left', column_width=None)]), _stub=<great_tables._gt_data.Stub object at 0x113e21d90>, _spanners=Spanners([]), _heading=Heading(title='Representative complete record', subtitle='Fields used to verify one supported result', preheader=None), _stubhead=None, _summary_rows=<great_tables._gt_data.SummaryRows object at 0x10f04fad0>, _summary_rows_grand=<great_tables._gt_data.SummaryRows object at 0x10f04fb00>, _source_notes=['The archive also stores lower, upper and point arrays, raw row identities, data hashes, warnings and native/source identities. This table selects a record before aggregation; validation recomputes every discrepancy from the retained arrays.'], _footnotes=[], _styles=[], _locale=<great_tables._gt_data.Locale object at 0x113e220f0>, _formats=[], _substitutions=[], _col_merge=[], _transforms=[], _options=Options(table_id=OptionsInfo(scss=False, category='table', type='value', value=None), table_caption=OptionsInfo(scss=False, category='table', type='value', value=None), table_width=OptionsInfo(scss=True, category='table', type='px', value='auto'), table_layout=OptionsInfo(scss=True, category='table', type='value', value='fixed'), table_margin_left=OptionsInfo(scss=True, category='table', type='px', value='auto'), table_margin_right=OptionsInfo(scss=True, category='table', type='px', value='auto'), table_background_color=OptionsInfo(scss=True, category='table', type='value', value='#FFFFFF'), table_additional_css=OptionsInfo(scss=False, category='table', type='values', value=[]), table_font_names=OptionsInfo(scss=False, category='table', type='values', value=['-apple-system', 'BlinkMacSystemFont', 'Segoe UI', 'Roboto', 'Oxygen', 'Ubuntu', 'Cantarell', 'Helvetica Neue', 'Fira Sans', 'Droid Sans', 'Arial', 'sans-serif']), table_font_size=OptionsInfo(scss=True, category='table', type='px', value='16px'), table_font_weight=OptionsInfo(scss=True, category='table', type='value', value='normal'), table_font_style=OptionsInfo(scss=True, category='table', type='value', value='normal'), table_font_color=OptionsInfo(scss=True, category='table', type='value', value='#333333'), table_font_color_light=OptionsInfo(scss=True, category='table', type='value', value='#FFFFFF'), table_border_top_include=OptionsInfo(scss=False, category='table', type='boolean', value=True), table_border_top_style=OptionsInfo(scss=True, category='table', type='value', value='solid'), table_border_top_width=OptionsInfo(scss=True, category='table', type='px', value='2px'), table_border_top_color=OptionsInfo(scss=True, category='table', type='value', value='#A8A8A8'), table_border_right_style=OptionsInfo(scss=True, category='table', type='value', value='none'), table_border_right_width=OptionsInfo(scss=True, category='table', type='px', value='2px'), table_border_right_color=OptionsInfo(scss=True, category='table', type='value', value='#D3D3D3'), table_border_bottom_include=OptionsInfo(scss=False, category='table', type='boolean', value=True), table_border_bottom_style=OptionsInfo(scss=True, category

Run from the repository root using the matching release extension and repository Python environment:

```bash
PYTHONPATH=python:. .venv/bin/python benches/audit_conformal_response_units.py \
  --phase corrected --output /tmp/conformal-response-units.json
PYTHONPATH=python:. .venv/bin/python -m benches.summarize_conformal_response_units
PYTHONPATH=python:. .venv/bin/python -m pytest tests/test_conformal_response_units_652.py -q
```

The runner refuses to overwrite evidence. The archived baseline uses production source from main commit `64a13b1`; the protocol was committed in `2edc1f3`, and the reference runner in `e139635` before measurement. Both phases use the identical release extension; its source digest and build identity are retained independently of the Python source hashes. The correction is reviewed in the MR linked from [issue #652](https://github.com/fcela/mboostrs/issues/652).